# Import necessary libraries

In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report


# STEP 1: LOAD DATA


In [4]:
file_path = "/content/train.csv"

# Read the CSV file with delimiter ';'
df = pd.read_csv(file_path, delimiter=";")

# Display the first few rows
print("Initial Data Overview:")
print(df.head())


Initial Data Overview:
   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  


# STEP 2: FEATURE ENGINEERING

In [5]:
# Conversion Rate Calculation

# Convert 'y' (target variable) to binary: 1 for 'yes' (converted), 0 for 'no'
df['conversion_binary'] = df['y'].apply(lambda x: 1 if x == 'yes' else 0)

# Compute conversion rate per contact attempt
df['conversion_rate'] = df['conversion_binary'] / df['campaign']

# Display first few rows with conversion rate
print("\n Conversion Rate Computation:")
print(df[['y', 'campaign', 'conversion_rate']].head())

# ✅ Sorting by Conversion Rate
df_sorted = df.sort_values(by='conversion_rate', ascending=False)

# Display the top records with highest conversion rate
print("\n Top Conversion Rate Records:")
print(df_sorted[['y', 'campaign', 'conversion_rate']].head())


 Conversion Rate Computation:
    y  campaign  conversion_rate
0  no         1              0.0
1  no         1              0.0
2  no         1              0.0
3  no         1              0.0
4  no         1              0.0

 Top Conversion Rate Records:
         y  campaign  conversion_rate
45204  yes         1              1.0
45203  yes         1              1.0
45202  yes         1              1.0
45185  yes         1              1.0
45178  yes         1              1.0


In [7]:
# Define the best time to contact each job category
contact_time_mapping = {
    "student": "6-8pm",
    "retired": "12-2pm",
    "unemployed": "12-2pm",
    "housemaid": "2-4pm",
    "admin.": "4-5pm",
    "management": "4-5pm",
    "entrepreneur": "4-5pm",
    "blue-collar": "4-5pm",
    "self-employed": "4-5pm",
    "technician": "4-5pm",
    "services": "4-5pm",
    "unknown": "4-5pm"
}

# Map the job category to best contact time
df["best_contact_time"] = df["job"].map(contact_time_mapping)

# Display sample records
print("\n Best Contact Time per Job:")
print(df[["job", "best_contact_time"]].head())


 Best Contact Time per Job:
            job best_contact_time
0    management             4-5pm
1    technician             4-5pm
2  entrepreneur             4-5pm
3   blue-collar             4-5pm
4       unknown             4-5pm


In [9]:
# Fatigue score computation

# Define decay factor based on campaign & previous contacts
df["decay_factor"] = df.apply(lambda row: 0.7 if (row["campaign"] + row["previous"]) > 5 else 0.5, axis=1)

# Compute Fatigue Score
df["fatigue_score"] = (df["campaign"] + df["previous"]) * df["decay_factor"]

# Display fatigue score computation results
print("\n Fatigue Score Computation:")
print(df[["campaign", "previous", "decay_factor", "fatigue_score"]].head())


 Fatigue Score Computation:
   campaign  previous  decay_factor  fatigue_score
0         1         0           0.5            0.5
1         1         0           0.5            0.5
2         1         0           0.5            0.5
3         1         0           0.5            0.5
4         1         0           0.5            0.5


In [10]:
# Customer Segmentation
# Encode Categorical Features
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Apply Label Encoding for categorical columns
for col in categorical_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

# Normalize Numerical Features
scaler = StandardScaler()
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Perform K-Means Clustering (5 Segments)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(df[numeric_cols])

# Define customer segment descriptions
cluster_mapping = {
    0: "Mid-High Income Males with Dependents, Strong Banking Relationship",
    1: "Young, Low-Income Females with Shortest Tenure & Low Credit",
    2: "Older, Low-Income Females with Strong Banking Relationship & High Utilisation",
    3: "Mid-Income Graduates with High Spending & Transactions",
    4: "Educated, Single Individuals with High Credit & Low Utilisation"
}

# Map clusters to descriptions
df['customer_segment'] = df['Cluster'].map(cluster_mapping)

# Train Models to Predict Customer Segments
X = df.drop(columns=['Cluster', 'customer_segment'])
y = df['Cluster']

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Decision Tree Classifier
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print(f"\n Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.2f}")
print(classification_report(y_test, y_pred_dt))

# Train Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f"\n Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.2f}")
print(classification_report(y_test, y_pred_rf))

# Train XGBoost Classifier
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
print(f"\n XGBoost Accuracy: {accuracy_score(y_test, y_pred_xgb):.2f}")
print(classification_report(y_test, y_pred_xgb))

# Apply Best Model (XGBoost) to Full Dataset
df['Predicted_Segment'] = xgb.predict(X)
df['customer_segment'] = df['Predicted_Segment'].map(cluster_mapping)

# Save the segmented customer data
df[['age', 'job', 'marital', 'education', 'customer_segment']].to_csv("customer_segments.csv", index=False)
print("Customer Segmentation Completed & Saved!")


 Decision Tree Accuracy: 0.92
              precision    recall  f1-score   support

           0       0.84      0.99      0.91      3604
           1       0.98      0.30      0.46       785
           2       1.00      0.94      0.97      2391
           3       0.99      1.00      1.00      1070
           4       0.95      0.98      0.96      1193

    accuracy                           0.92      9043
   macro avg       0.95      0.84      0.86      9043
weighted avg       0.93      0.92      0.90      9043


 Random Forest Accuracy: 0.99
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3604
           1       1.00      0.99      0.99       785
           2       1.00      0.99      1.00      2391
           3       0.99      1.00      1.00      1070
           4       0.99      0.99      0.99      1193

    accuracy                           0.99      9043
   macro avg       0.99      0.99      0.99      9043
weighted avg   

# STEP 3: MULTI-ARMED BANDIT (MAB) MODEL

In [12]:
# Simulated dataset
np.random.seed(42)

# Simulated data: 60 days of conversion data for treatment and control groups
n_days = 60
treatment_group = np.random.normal(loc=1.12, scale=0.02, size=n_days)  # Slightly higher mean
control_group = np.random.normal(loc=1.10, scale=0.02, size=n_days)   # Lower mean

# DataFrame to store results
results = pd.DataFrame({
    "day": np.arange(n_days),
    "treatment_mean": treatment_group,
    "control_mean": control_group
})

# Bootstrap resampling for statistical significance
n_iterations = 10000
diff_means = []

for _ in range(n_iterations):
    sample_treatment = np.random.choice(treatment_group, size=len(treatment_group), replace=True)
    sample_control = np.random.choice(control_group, size=len(control_group), replace=True)
    diff_means.append(np.mean(sample_treatment) - np.mean(sample_control))

# Compute 95% Confidence Interval
ci_lower, ci_upper = np.percentile(diff_means, [2.5, 97.5])
print(f"95% Confidence Interval for Difference in Means: [{ci_lower}, {ci_upper}]")

# --------------------------------------------
# IMPROVED REWARD FUNCTION
# --------------------------------------------
alpha, beta, gamma = 1.0, 0.5, 0.3  # Weights for different reward components

def calculate_reward(conversion_rate, clv, fatigue_score):
    return (alpha * conversion_rate) + (beta * clv) - (gamma * fatigue_score)

# Simulated additional factors
customer_lifetime_value = np.random.uniform(50, 200, n_days)  # CLV per user
contact_attempts = np.random.randint(1, 5, n_days)  # Number of contacts
fatigue_score = np.exp(-0.1 * contact_attempts)  # Decay function for fatigue

# Compute rewards for each day
results["reward_treatment"] = calculate_reward(results["treatment_mean"], customer_lifetime_value, fatigue_score)
results["reward_control"] = calculate_reward(results["control_mean"], customer_lifetime_value, fatigue_score)

# --------------------------------------------
# BAYESIAN THOMPSON SAMPLING (MAB)
# --------------------------------------------
alpha_treatment, beta_treatment = 1, 1
alpha_control, beta_control = 1, 1

for _ in range(1000):
    treatment_sample = np.random.beta(alpha_treatment, beta_treatment)
    control_sample = np.random.beta(alpha_control, beta_control)

    if treatment_sample > control_sample:
        # Choose treatment group (MAB)
        alpha_treatment += 1  # Increase success count
    else:
        # Choose control group (Rule-based)
        beta_control += 1  # Increase failure count

# Display Final Summary
print("\nEnhanced A/B Test Results:")
print(results.describe())

95% Confidence Interval for Difference in Means: [0.010337339628789488, 0.023640834721986435]

Enhanced A/B Test Results:
             day  treatment_mean  control_mean  reward_treatment  \
count  60.000000       60.000000     60.000000         60.000000   
mean   29.500000        1.116907      1.099926         61.071434   
std    17.464249        0.018170      0.018867         21.385036   
min     0.000000        1.080807      1.047605         25.938374   
25%    14.750000        1.106247      1.090304         42.031960   
50%    29.500000        1.115401      1.099388         59.799081   
75%    44.250000        1.128119      1.107445         76.781603   
max    59.000000        1.157046      1.149265        100.599015   

       reward_control  
count       60.000000  
mean        61.054453  
std         21.384283  
min         25.945040  
25%         41.997384  
50%         59.771643  
75%         76.786144  
max        100.554722  
